## Surface Animation  with  slider ##

Solution for week 15 practice in Data Visualization - version 4

In [1]:
import numpy as np
import plotly.offline as off

In [2]:
TIME_STEPS = 37

In [3]:
def create_data(x, y, z0, zmin, zmax):
    """
    Input: 2D numpy arrays x, y, z0 that defines initial surface
    
    Output: List consisting of a single dictionary defining this surface
    """

    return [dict(type='surface',
                 x=x, 
                 y=y, 
                 z=z0, 
                 cmin=zmin,
                 cmax=zmax)]

In [4]:
def create_frames(z):
    """
    Input: 3D numpy array z that defines sequence of height maps
    
    Output: List of dictionaries corresponding to frames in animation
    """
    
    return [dict(data=[dict(z=z[k])],
                 name='frame{}'.format(k)) 
            for k  in  range(TIME_STEPS)]
            

In [5]:
def create_slider(t):
    """
    Input: 1D numpy array t of times
    
    Output: List consisting a single dictionary that defines a slider
    This dictionary includes a "steps" key whose value is a list of dictionaries
    corresponding to each frame in the animation
    """
    
    return [dict(steps=[dict(method='animate',
                             args=[['frame{}'.format(k) ],
                                   dict(mode= 'immediate',
                                        frame= dict( duration=50, redraw=True ),
                                        transition=dict( duration= 0))],
                             label='{:.2f}'.format(t[k])) 
                        for k in range(TIME_STEPS)], 
                 transition= dict(duration= 0 ),
                 x=0,#slider starting position  
                 y=0, 
                 currentvalue=dict(font=dict(size=12), 
                                   prefix='Time: ', 
                                   visible=True, 
                                   xanchor= 'center'),
                 len=1.0)]

In [6]:
def create_button():
    """
    Output: List consisting of single dictionary defining a button
    """
    
    return [dict(type='buttons', 
                 showactive=False,
                 y=0,
                 x=1.15,
                 xanchor='right',
                 yanchor='top',
                 pad=dict(t=0, r=10),
                 buttons=[dict(label='Play',
                               method='animate',
                               args=[None, 
                                     dict(frame=dict(duration=30, 
                                                     redraw=True),
                                          transition=dict(duration=0),
                                          fromcurrent=True,
                                          mode='immediate')])])]

In [7]:
def create_layout(t, zmin, zmax):
    """
    Input: 1D numpy array t of times, float zmin, zmax
    
    Output: Dictionary corresponding to the layout of the animation
    Uses create_button() and create_slider() as compute values for the "updatemenus"
    and "sliders" keys, respectively
    """
    
    return dict(title='Animating a 2d wave',
                autosize=False,
                width=600,
                height=600,
                showlegend=False,
                scene=dict(camera = dict(eye=dict(x=1.25, y=0.9, z=1.1)),
                           aspectratio=dict(x=1, y=1, z=0.5),
                           zaxis=dict(range=[zmin, zmax])),
                updatemenus=create_button(),
                sliders=create_slider(t))

In [8]:
def run_animation():
    """ Run the animation code """
    
    # Define the surface
    u = np.linspace(-8, 8, 100)
    x, y = np.meshgrid(u, u)
    r = np.sqrt(x**2+y**2)
    t = np.linspace(0, np.pi*2, TIME_STEPS)
    z = np.array([(np.cos(r + s) * np.exp(-r/5)) for s in t])
    zmin=np.min(z)
    zmax=np.max(z)
     
    # Create and plot the figure
    fig=dict(data=create_data(x, y, z[0], zmin, zmax),
             frames=create_frames(z),
             layout=create_layout(t, zmin, zmax))
    off.plot(fig, filename="animation.html", validate=False) 

    #off.init_notebook_mode(connected=True)
    #off.iplot(fig, validate=False)

    #validate=False is needed since there is an issue in offline.iplot:
    #https://github.com/plotly/plotly.py/issues/702

In [9]:
run_animation()